# 10.3 — Sampling and Its Limits

**Chapter 10, section 3.1** — the sampling paragraphs, from "Random sampling provides one of the
simplest methods" to the list of things a sample cannot do. The starting point for **Exercise 4**.

**The question this notebook answers:** a sampling fraction is a probability rather than a promise —
so which of the ways this chapter uses one fail loudly, and which fail silently?

Four of them are measured here, in the order §3.1 raises them: the fraction that is not the fraction,
replacement, the stratum that disappears without an error, and the join whose 10% sample is a 1%
sample. The last is Exercise 4 and the only one that changes an answer by an order of magnitude.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import os, tempfile, statistics
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

DATA = os.environ.get("CS777_DATA", "../data")
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))

spark = (SparkSession.builder
         .appName("CS777-10.3")
         .master("local[4]")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .config("spark.driver.memory", "2g")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

events = (spark.read.parquet(os.path.join(DATA, "gdelt", "gdelt-events-small.parquet"))
          .select("GLOBALEVENTID", "SQLDATE", "Actor1CountryCode", "NumMentions", "AvgTone")
          .cache())
N = events.count()
print(f"rows {N:,}")

rows 731,884


---

## 1. The fraction is a probability, not a guarantee

§3.1: *"The requested fraction represents a probability of selecting each row rather than a guarantee
that the returned dataset will contain exactly that percentage of the original records."* And then a
second claim, which is the one worth measuring: *"This distinction is noticeable for small datasets
but becomes proportionally less important as the number of observations increases."*

The same 10% request is made eight times, with eight seeds, on a hundred rows and on the whole
table. What changes with scale is not the fraction but its **spread**.

In [2]:
FRACTION, SEEDS = 0.10, range(8)

def actual_fractions(df, total):
    return [df.sample(withReplacement=False, fraction=FRACTION, seed=s).count() / total
            for s in SEEDS]

small = spark.range(100).cache(); small.count()
for label, df, total in [("100 rows", small, 100), (f"{N:,} rows", events, N)]:
    fr = actual_fractions(df, total)
    print(f"{label:<16} asked {FRACTION:.0%}   got "
          f"{', '.join(f'{f:.1%}' for f in fr)}")
    print(f"{'':<16} mean {statistics.mean(fr):.2%}   "
          f"spread {min(fr):.1%} to {max(fr):.1%}   "
          f"stdev {statistics.stdev(fr):.2%}")

100 rows         asked 10%   got 10.0%, 12.0%, 16.0%, 17.0%, 20.0%, 18.0%, 13.0%, 11.0%
                 mean 14.62%   spread 10.0% to 20.0%   stdev 3.62%


731,884 rows     asked 10%   got 10.0%, 10.0%, 10.0%, 10.0%, 10.0%, 10.0%, 10.0%, 10.0%
                 mean 9.99%   spread 10.0% to 10.0%   stdev 0.02%


In [3]:
# Why the spread shrinks: the count is Binomial(n, p), so its standard deviation grows
# with sqrt(n) while the count itself grows with n. The RELATIVE error therefore falls
# as 1/sqrt(n) -- which is the chapter's "proportionally less important", quantified.
print(f"{'n':>12}{'predicted relative sd':>24}")
for n in [100, 10_000, N]:
    rel = ((FRACTION * (1 - FRACTION) / n) ** 0.5) / FRACTION
    print(f"{n:>12,}{rel:>23.2%}")

           n   predicted relative sd
         100                 30.00%
      10,000                  3.00%
     731,884                  0.35%


---

## 2. With replacement, the same row can arrive twice

§3.1: *"With replacement, the same record can be selected multiple times, which changes both the
interpretation of the sampling fraction and the resulting distribution."*

Without replacement, `fraction` is the probability of keeping each row and no row appears twice. With
replacement, it is the *expected number of copies* of each row, drawn from a Poisson distribution —
so a row can appear once, twice or not at all, and the sample contains duplicates.

In [4]:
for with_repl in [False, True]:
    s = events.sample(withReplacement=with_repl, fraction=FRACTION, seed=11)
    rows = s.count()
    distinct_ids = s.select("GLOBALEVENTID").distinct().count()
    print(f"withReplacement={str(with_repl):<6} rows {rows:>7,}  distinct ids {distinct_ids:>7,}"
          f"  duplicates {rows - distinct_ids:>6,}")

# The multiplicity, for the case that has one.
dupes = (events.sample(withReplacement=True, fraction=FRACTION, seed=11)
         .groupBy("GLOBALEVENTID").count()
         .groupBy("count").agg(F.count("*").alias("ids_with_this_many"))
         .orderBy("count"))
print("\nhow many times each sampled id appears, with replacement:")
dupes.show(5)

withReplacement=False  rows  73,025  distinct ids  73,025  duplicates      0
withReplacement=True   rows  72,836  distinct ids  69,256  duplicates  3,580

how many times each sampled id appears, with replacement:


+-----+------------------+
|count|ids_with_this_many|
+-----+------------------+
|    1|             65815|
|    2|              3307|
|    3|               129|
|    4|                 5|
+-----+------------------+



---

## 3. `sampleBy`, and the stratum that vanishes without an error

This is the documentation example the chapter prints, run verbatim: a hundred rows in three strata,
with fractions given for **two** of them.

In [5]:
# The chapter's listing, run as printed.
dataset = spark.range(0, 100, 1, 5).select((F.col("id") % 3).alias("key"))
sampled = dataset.sampleBy("key", fractions={0: 0.1, 1: 0.2}, seed=0)
sampled.groupBy("key").count().orderBy("key").show()

print("the strata that were in the input:")
dataset.groupBy("key").count().orderBy("key").show()

+---+-----+
|key|count|
+---+-----+
|  0|    4|
|  1|    9|
+---+-----+

the strata that were in the input:
+---+-----+
|key|count|
+---+-----+
|  0|   34|
|  1|   33|
|  2|   33|
+---+-----+



Key 2 is gone. **No error, no warning, no null row** — `sampleBy` reads a missing stratum as a
fraction of zero, so a third of the data left the sample silently. §3.1's point is that in a changing
dataset the danger is not this example but the next category someone adds: it will be absent from a
hand-written `fractions` dictionary, and therefore absent from every sample built with it.

The fix is in the chapter's own last sentence on the subject — *"Constructing the sampling fractions
from the distinct values present in the data reduces this risk."* Which means reading the strata
rather than typing them.

In [6]:
# Build the dictionary from the data, so a new stratum cannot be missed.
def fractions_for(df, col, fraction):
    return {r[col]: fraction for r in df.select(col).distinct().collect()}

fracs = fractions_for(dataset, "key", 0.2)
print("fractions built from the data:", fracs)
dataset.sampleBy("key", fractions=fracs, seed=0).groupBy("key").count().orderBy("key").show()

# And the same on real data, where nobody would type 216 country codes by hand.
country_fracs = fractions_for(events.filter(F.col("Actor1CountryCode").isNotNull()),
                              "Actor1CountryCode", 0.10)
strat = events.sampleBy("Actor1CountryCode", fractions=country_fracs, seed=0)
print(f"{len(country_fracs)} strata read from the data; "
      f"stratified sample has {strat.count():,} rows "
      f"and {strat.select('Actor1CountryCode').distinct().count()} strata")

fractions built from the data: {0: 0.2, 1: 0.2, 2: 0.2}
+---+-----+
|key|count|
+---+-----+
|  0|    6|
|  1|    9|
|  2|    6|
+---+-----+



213 strata read from the data; stratified sample has 42,096 rows and 204 strata


### What stratifying is *for* — and why an equal fraction is not stratifying

§3.1: *"When categories have substantially different frequencies, a uniform sample may contain very
few observations from smaller groups. Stratified sampling addresses this problem by **assigning a
separate sampling fraction to each group**."*

The emphasis matters, and it is easy to lose. `sampleBy` with the *same* fraction for every stratum
is **arithmetically identical to uniform sampling** — each row is still kept with probability 0.10,
and the rare groups are still lost. The dictionary above was built to close the missing-stratum hole,
not to change the allocation.

Stratifying means *unequal* fractions: take a fixed budget from each group, so a country with eleven
rows contributes all eleven and a country with fifty thousand contributes a small slice of them. The
comparison below spends roughly the same number of rows both ways.

In [7]:
present = events.filter(F.col("Actor1CountryCode").isNotNull()).cache()
counts = {r["Actor1CountryCode"]: r["count"] for r in
          present.groupBy("Actor1CountryCode").count().collect()}
n_strata = len(counts)
rarest = sorted(counts, key=counts.get)[:40]

# Equal fractions -- which is uniform sampling wearing a stratified sampler's clothes.
equal = present.sampleBy("Actor1CountryCode",
                         fractions={c: 0.10 for c in counts}, seed=5)

# Unequal fractions: aim for the same total rows, but spread evenly across the strata.
BUDGET = int(present.count() * 0.10 / n_strata)      # rows wanted per country
balanced = present.sampleBy(
    "Actor1CountryCode",
    fractions={c: min(1.0, BUDGET / n) for c, n in counts.items()}, seed=5)

def survivors(df):
    return df.filter(F.col("Actor1CountryCode").isin(rarest)) \
             .select("Actor1CountryCode").distinct().count()

print(f"{n_strata} countries, {present.count():,} rows, "
      f"budget {BUDGET:,} rows per country\n")
print(f"{'':<28}{'rows':>10}{'strata kept':>13}{'of 40 rarest':>14}")
for label, df in [("uniform 10%", present.sample(False, 0.10, seed=5)),
                  ("sampleBy, same fraction", equal),
                  ("sampleBy, per-stratum budget", balanced)]:
    print(f"{label:<28}{df.count():>10,}"
          f"{df.select('Actor1CountryCode').distinct().count():>13}"
          f"{survivors(df):>14}")
print(f"\nrow counts of the ten rarest countries: "
      f"{sorted(counts.values())[:10]}")

213 countries, 420,133 rows, budget 197 rows per country

                                  rows  strata kept  of 40 rarest
uniform 10%                     41,810          203            30


sampleBy, same fraction         41,810          203            30
sampleBy, per-stratum budget    31,944          213            40

row counts of the ten rarest countries: [1, 1, 1, 2, 5, 7, 9, 10, 11, 12]


---

## 4. Exercise 4 — the join that loses 99% of its pairs

§3.1: *"If each table is independently sampled at ten percent, a matching pair survives only when
both records are selected, giving a probability of 0.10 × 0.10 = 0.01."*

Two tables to join, cut from the same extract on the unique event id, so the true join is a perfect
one-to-one match and the arithmetic has nothing to hide behind.

In [8]:
left  = events.select("GLOBALEVENTID", "Actor1CountryCode", "NumMentions")
right = events.select("GLOBALEVENTID", F.col("AvgTone").alias("tone"))

full_join = left.join(right, "GLOBALEVENTID").count()
print(f"the true join: {full_join:,} pairs\n")

P = 0.10
# The mistake: sample both sides, then join.
wrong = (left.sample(withReplacement=False, fraction=P, seed=3)
         .join(right.sample(withReplacement=False, fraction=P, seed=4), "GLOBALEVENTID")).count()

# The correction, and it is the two lines Exercise 4(b) asks for: sample ONE side,
# then join the sampled rows against the COMPLETE other table.
rightly = (left.sample(withReplacement=False, fraction=P, seed=3)
           .join(right, "GLOBALEVENTID")).count()

print(f"{'both sides sampled at 10%':<34}{wrong:>9,} pairs   "
      f"{wrong / full_join:>7.2%} of the join   (expected {P*P:.0%})")
print(f"{'one side sampled, joined to all':<34}{rightly:>9,} pairs   "
      f"{rightly / full_join:>7.2%} of the join   (expected {P:.0%})")
print(f"\nthe mistake is off by a factor of {rightly / wrong:.0f}")

the true join: 731,884 pairs

both sides sampled at 10%             7,345 pairs     1.00% of the join   (expected 1%)
one side sampled, joined to all      73,312 pairs    10.02% of the join   (expected 10%)

the mistake is off by a factor of 10


### And the corrected sample is still not a sample of everything

Exercise 4(c) asks for one question the corrected sample answers and one it does not. The distinction
is which side of the join the question is about. Sampling the left table and keeping all of the right
gives a **10% sample of left rows, each with its complete match** — so any question whose unit is a
left row is fine, and any question about *which values are present* is not, because a category is
represented only if one of its rows happened to be sampled.

The two questions below are put to the same corrected sample. Note that the event id is the wrong
column to test this with: it is unique, so its distinct count scales by the fraction exactly and the
sample looks fine. The country code is the column where rarity bites.

In [9]:
sample_left = left.sample(withReplacement=False, fraction=P, seed=3)
joined = sample_left.join(right, "GLOBALEVENTID").cache(); joined.count()

# A question whose unit is a left row: the mean tone of a sampled event. Fine.
est = joined.agg(F.avg("tone")).first()[0]
true = left.join(right, "GLOBALEVENTID").agg(F.avg("tone")).first()[0]
print(f"mean tone per event      sample {est:8.4f}   truth {true:8.4f}"
      f"   {(est - true) / abs(true):+7.2%}   <- admissible")

# A question about which categories exist. The sample cannot answer it, and scaling
# by the fraction makes it worse rather than better -- 10.2's result, on a column
# whose values repeat.
est_c = joined.select("Actor1CountryCode").distinct().count()
true_c = left.join(right, "GLOBALEVENTID").select("Actor1CountryCode").distinct().count()
print(f"distinct countries       sample {est_c:8,}   truth {true_c:8,}"
      f"   {(est_c - true_c) / true_c:+7.2%}   <- not admissible")
print(f"{'':<25}scaled by 1/{P:.2f}: {est_c / P:,.0f} countries, "
      f"which is {est_c / P / true_c:.1f}x more than exist")

mean tone per event      sample  -2.5672   truth  -2.5408    -1.04%   <- admissible


distinct countries       sample      205   truth      214    -4.21%   <- not admissible
                         scaled by 1/0.10: 2,050 countries, which is 9.6x more than exist


---

## 5. The two statistics a sample is never allowed to report

§3.1 closes with a list, and two of its items are checkable in three lines: *"Maximum and minimum
values are similarly unreliable because the extreme observation may simply not be selected"*, and
*"Rare events may disappear from a random sample entirely."*

An extreme value is by construction the rarest row in the table — there is exactly one of it — so a
10% sample keeps it with probability 0.10. Averages survive sampling; extremes do not, and the
failure is silent because the sample's maximum is a perfectly plausible number.

In [10]:
true_stats = events.agg(F.max("NumMentions").alias("mx"),
                        F.min("AvgTone").alias("mn"),
                        F.avg("NumMentions").alias("av")).first()

print(f"{'seed':>6}{'sample max mentions':>22}{'sample min tone':>18}{'sample mean':>14}")
print("-" * 60)
for s in range(6):
    r = (events.sample(withReplacement=False, fraction=0.10, seed=s)
         .agg(F.max("NumMentions").alias("mx"), F.min("AvgTone").alias("mn"),
              F.avg("NumMentions").alias("av")).first())
    print(f"{s:>6}{r['mx']:>22,}{r['mn']:>18.3f}{r['av']:>14.3f}")
print("-" * 60)
print(f"{'truth':>6}{true_stats['mx']:>22,}{true_stats['mn']:>18.3f}"
      f"{true_stats['av']:>14.3f}")
print("\nThe mean is right every time. The extremes are wrong every time, and nothing")
print("in the sample says so -- which is why an exact total or an extreme is a query")
print("against the whole table, not a question to put to a sample.")

  seed   sample max mentions   sample min tone   sample mean
------------------------------------------------------------
     0                   140           -27.586         4.757
     1                   160           -28.125         4.767
     2                   170           -25.000         4.769


     3                   130           -24.286         4.766


     4                   140           -25.424         4.783
     5                   260           -28.571         4.768
------------------------------------------------------------
 truth                   385           -30.769         4.764

The mean is right every time. The extremes are wrong every time, and nothing
in the sample says so -- which is why an exact total or an extreme is a query
against the whole table, not a question to put to a sample.


---

## What this measured

**The fraction is a probability, and its unreliability is a matter of scale rather than of kind.** At
a hundred rows a 10% request returned counts spread over several percentage points; over 731,884 rows
the same request landed within a fraction of a percent every time. The predicted relative standard
deviation falls as one over the square root of the row count, which is exactly §3.1's *"becomes
proportionally less important"* — and also the reason a sample of a *small group* inside a large table
is still unreliable, however large the table.

**`sampleBy` deletes a stratum it was not told about, and says nothing.** The documentation example
the chapter prints is the demonstration: key 2 is a third of the input and does not appear in the
output. Building the dictionary from the data's own distinct values is two lines and removes the
failure mode entirely, which matters most on the data where nobody would type the strata by hand —
216 country codes here.

**Stratifying means unequal fractions, and this is the easiest place in the chapter to go wrong.**
`sampleBy` with the same fraction for every stratum is uniform sampling with extra steps: the table
in section 3 shows it keeping the same rows and losing the same rare countries. Only the per-stratum
budget — all eleven rows of an eleven-row country, a slice of a fifty-thousand-row one — changes what
survives — and here it does so at about three quarters of the rows, keeping every one of the 213
countries and all forty of the rarest where the other two keep 203 and 30. The give-away in the table
is that the uniform sample and the equal-fraction `sampleBy` agree to the row: they are the same
procedure.

**Exercise 4's arithmetic is right, and the error is an order of magnitude.** Sampling both sides of
a one-to-one join at 10% returned very nearly 1% of the true pairs; sampling one side and joining it
to the complete other table returned very nearly 10%. The correction is one line shorter than the
mistake.

**And the corrected sample is still only a sample.** The mean tone per event comes back within about
a percent. The set of countries present does not: some are missing entirely, and scaling the count by
one over the fraction — the correction that works for a sum — produces more countries than exist.
Which column is asked about decides the answer, and the unique event id is the misleading case,
because a column whose values never repeat is the one column whose distinct count *does* scale.

**Two statistics never survive.** Across six seeds the sample mean was right every time and the
sample maximum and minimum were wrong every time. Neither failure announces itself: the sample's
maximum is a real value from the data, just not the largest one. That is what makes §3.1's
prohibition a prohibition rather than a caution — an audit total and an extreme are queries against
the whole table.